# Semiparametric Notebook: Dynamic DML Pipeline

This notebook gives a small simulation check for `DML_dynamic`, the two-period dynamic treatment effect estimator. The target below is `E[Y(1, 1)]`.

## 1) Imports and setup

In [14]:
# ---- Limit BLAS/OpenMP threads BEFORE importing heavy libs ----
import os as os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

# ---- Standard libs ----
import sys
import time
import platform
from pathlib import Path

# ---- Third-party ----
import numpy as np
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from threadpoolctl import threadpool_limits
import torch
import torch.nn as nn

# Keep native libraries to 1 thread (NumPy/SciPy/BLAS/OpenMP)
threadpool_limits(1)

# ---- Local repo imports ----
cwd = Path.cwd().resolve()
repo_root = cwd
if not (repo_root / "nnpiv").exists():
    for parent in cwd.parents:
        if (parent / "nnpiv").exists():
            repo_root = parent
            break
sys.path.insert(0, str(repo_root))

from nnpiv.ensemble import EnsembleIV
from nnpiv.linear import sparse_l2vsl2
from nnpiv.neuralnet import AGMM
from nnpiv.rkhs import RKHSIVL2
from nnpiv.semiparametrics import DML_dynamic
from nnpiv.tsls import tsls


def seed_everything(seed: int = 123) -> None:
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.set_num_threads(1)
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass


seed_everything(123)


def print_resources():
    print("=== Compute resources ===")
    print(f"Python: {sys.version.split()[0]}")
    print(f"NumPy: {np.__version__}")
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"CPU cores: {os.cpu_count()}")
    print(f"Platform: {platform.platform()}")
    print("Thread caps (env):")
    for k in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"]:
        print(f"  {k}={os.environ.get(k, 'unset')}")
    print("=========================\n")


def summarize_dml_result(name: str, result, elapsed: float):
    theta, var, ci = result
    theta = np.atleast_1d(theta).astype(float)
    var = np.atleast_1d(var).astype(float)
    se = np.sqrt(var)
    ci = np.asarray(ci, dtype=float)

    print(f"[{name}] time={elapsed:.2f}s")
    print(f"  theta : {theta[0]:.4f}")
    print(f"  SE    : {se[0]:.4f}")
    print(f"  95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")
    print("")

## 2) Resource check

In [15]:
print_resources()

=== Compute resources ===
Python: 3.14.4
NumPy: 2.3.2
PyTorch: 2.11.0
CUDA available: False
CPU cores: 18
Platform: macOS-26.5.1-arm64-arm-64bit-Mach-O
Thread caps (env):
  OMP_NUM_THREADS=1
  OPENBLAS_NUM_THREADS=1
  MKL_NUM_THREADS=1
  VECLIB_MAXIMUM_THREADS=1
  NUMEXPR_NUM_THREADS=1



## 3) Simulate a two-period dynamic treatment design

The treatment propensities are logistic and the outcome/state regressions are linear, so `tsls` and `LogisticRegression` are intentionally well specified here. This makes the notebook a quick smoke test for the score and cross-fitting pipeline.

In [16]:
def simulate_dynamic_data(n: int = 1000, seed: int = 123):
    rng = np.random.default_rng(seed)

    X1 = rng.normal(size=(n, 2))
    p1 = expit(0.2 + 0.5 * X1[:, 0] - 0.25 * X1[:, 1])
    D1 = rng.binomial(1, p1)

    e21 = rng.normal(size=n)
    e22 = rng.normal(size=n)
    X2 = np.column_stack([
        0.4 * D1 + 0.5 * X1[:, 0] + e21,
        -0.2 * D1 + 0.3 * X1[:, 1] + e22,
    ])

    p2 = expit(-0.1 + 0.6 * D1 + 0.4 * X2[:, 0] - 0.2 * X1[:, 0])
    D2 = rng.binomial(1, p2)

    eps_y = rng.normal(size=n)
    Y = (1.0 + 0.7 * D1 + 1.2 * D2 + 0.3 * D1 * D2
         + 0.8 * X1[:, 0] - 0.5 * X1[:, 1]
         + 0.6 * X2[:, 0] + 0.3 * X2[:, 1] + eps_y)

    return Y, D1, D2, X1, X2


def true_dynamic_mean(d1: int = 1, d2: int = 1, n_mc: int = 1_000_000, seed: int = 123):
    rng = np.random.default_rng(seed)
    X1 = rng.normal(size=(n_mc, 2))
    e21 = rng.normal(size=n_mc)
    e22 = rng.normal(size=n_mc)
    X2_d = np.column_stack([
        0.4 * d1 + 0.5 * X1[:, 0] + e21,
        -0.2 * d1 + 0.3 * X1[:, 1] + e22,
    ])
    y_d = (1.0 + 0.7 * d1 + 1.2 * d2 + 0.3 * d1 * d2
           + 0.8 * X1[:, 0] - 0.5 * X1[:, 1]
           + 0.6 * X2_d[:, 0] + 0.3 * X2_d[:, 1])
    return y_d.mean()


Y, D1, D2, X1, X2 = simulate_dynamic_data(n=2000, seed=123)
TRUE_PARAM = true_dynamic_mean(d1=1, d2=1)

print(f"Sample size: {Y.shape[0]}")
print(f"Share D1=1: {D1.mean():.3f}")
print(f"Share D2=1: {D2.mean():.3f}")
print(f"True E[Y(1,1)] from large Monte Carlo: {TRUE_PARAM:.4f}")

Sample size: 2000
Share D1=1: 0.552
Share D2=1: 0.582
True E[Y(1,1)] from large Monte Carlo: 3.3802


## 4) `DML_dynamic` for `E[Y(1,1)]`

In [17]:
DYNAMIC_RESULTS = []


def _get_learner(n_t: int, n_hidden: int = 32, p: float = 0.1) -> nn.Module:
    return nn.Sequential(
        nn.Dropout(p=p),
        nn.Linear(n_t, n_hidden),
        nn.LeakyReLU(),
        nn.Linear(n_hidden, 1),
    )


def _get_adversary(n_z: int, n_hidden: int = 32, p: float = 0.1) -> nn.Module:
    return nn.Sequential(
        nn.Dropout(p=p),
        nn.Linear(n_z, n_hidden),
        nn.LeakyReLU(),
        nn.Linear(n_hidden, 1),
    )


def run_and_report(label: str, dml_obj):
    t0 = time.perf_counter()
    result = dml_obj.dml()
    t1 = time.perf_counter()
    summarize_dml_result(label, result, t1 - t0)
    DYNAMIC_RESULTS.append((label, result, t1 - t0))
    return result


# =========================================================
# 1) Sequential estimator (MR) with EnsembleIV
# =========================================================
dml_ensemble = DML_dynamic(
    Y, D1, D2, X1, X2,
    estimator="MR",
    treatment_path=(1, 1),
    nu_score="S-DRL",
    model1=[
        EnsembleIV(n_iter=100, max_abs_value=4),
        EnsembleIV(n_iter=100, max_abs_value=4),
    ],
    nn_1=[False, False],
    fitargs1=[None, None],
    prop_score=LogisticRegression(max_iter=2000),
    n_folds=3, n_rep=1,
    inner_n_jobs=1,
    random_seed=123,
    verbose=False,
)
res_ensemble = run_and_report("Sequential (MR) with EnsembleIV", dml_ensemble)


# =========================================================
# 2) Sequential estimator (MR) with TSLS
# =========================================================
dml_tsls = DML_dynamic(
    Y, D1, D2, X1, X2,
    estimator="MR",
    treatment_path=(1, 1),
    nu_score="S-DRL",
    model1=[tsls(), tsls()],
    nn_1=[False, False],
    fitargs1=[None, None],
    prop_score=LogisticRegression(max_iter=2000),
    n_folds=3, n_rep=1,
    inner_n_jobs=1,
    random_seed=123,
    verbose=False,
)
res_tsls = run_and_report("Sequential (MR) with TSLS", dml_tsls)


# =========================================================
# 3) Sequential estimator (MR) with RKHSIVL2
# =========================================================
dml_rkhs = DML_dynamic(
    Y, D1, D2, X1, X2,
    estimator="MR",
    treatment_path=(1, 1),
    nu_score="S-DRL",
    model1=[
        RKHSIVL2(kernel="rbf", gamma=0.1, delta_scale="auto", delta_exp=0.4),
        RKHSIVL2(kernel="rbf", gamma=0.1, delta_scale="auto", delta_exp=0.4),
    ],
    nn_1=[False, False],
    fitargs1=[None, None],
    prop_score=LogisticRegression(max_iter=2000),
    n_folds=3, n_rep=1,
    inner_n_jobs=1,
    random_seed=123,
    verbose=False,
)
res_rkhs = run_and_report("Sequential (MR) with RKHSIVL2", dml_rkhs)


# =========================================================
# 4) Sequential estimator (MR) with sparse_l2_l2
# =========================================================
dml_sparse = DML_dynamic(
    Y, D1, D2, X1, X2,
    estimator="MR",
    treatment_path=(1, 1),
    nu_score="S-DRL",
    model1=[
        sparse_l2vsl2(lambda_theta=0.01, B=100, n_iter=1000, tol=1e-3),
        sparse_l2vsl2(lambda_theta=0.01, B=100, n_iter=1000, tol=1e-3),
    ],
    nn_1=[False, False],
    fitargs1=[None, None],
    prop_score=LogisticRegression(max_iter=2000),
    n_folds=3, n_rep=1,
    inner_n_jobs=1,
    random_seed=123,
    verbose=False,
)
res_sparse = run_and_report("Sequential (MR) with sparse_l2_l2", dml_sparse)


# =========================================================
# 5) Sequential estimator (MR) with AGMM
# =========================================================
agmm_fitargs = {
    "learner_lr": 1e-4,
    "adversary_lr": 1e-4,
    "learner_l2": 1e-3,
    "adversary_l2": 1e-4,
    "adversary_norm_reg": 1e-3,
    "n_epochs": 20,
    "bs": 128,
    "train_learner_every": 1,
    "train_adversary_every": 1,
    "ols_weight": 0.0,
    "verbose": 0,
}

dml_agmm = DML_dynamic(
    Y, D1, D2, X1, X2,
    estimator="MR",
    treatment_path=(1, 1),
    nu_score="S-DRL",
    model1=[
        AGMM(
            _get_learner(X1.shape[1] + X2.shape[1]),
            _get_adversary(X1.shape[1] + X2.shape[1]),
        ),
        AGMM(
            _get_learner(X1.shape[1]),
            _get_adversary(X1.shape[1]),
        ),
    ],
    nn_1=[True, True],
    fitargs1=[agmm_fitargs, agmm_fitargs],
    prop_score=LogisticRegression(max_iter=2000),
    n_folds=3, n_rep=1,
    inner_n_jobs=1,
    random_seed=123,
    verbose=False,
)
res_agmm = run_and_report("Sequential (MR) with AGMM", dml_agmm)


print(f"Reference truth: {TRUE_PARAM:.4f}")

[Sequential (MR) with EnsembleIV] time=8.20s
  theta : 3.4275
  SE    : 2.6043
  95% CI: [3.3133, 3.5416]

[Sequential (MR) with TSLS] time=0.01s
  theta : 3.4736
  SE    : 2.3452
  95% CI: [3.3708, 3.5764]

[Sequential (MR) with RKHSIVL2] time=0.29s
  theta : 3.6502
  SE    : 12.6838
  95% CI: [3.0943, 4.2061]

[Sequential (MR) with sparse_l2_l2] time=0.05s
  theta : 3.4760
  SE    : 2.3588
  95% CI: [3.3726, 3.5793]

[Sequential (MR) with AGMM] time=0.38s
  theta : 3.4776
  SE    : 5.0876
  95% CI: [3.2546, 3.7006]

Reference truth: 3.3802
